1.Imprting & Data Loading

In [183]:
import pandas as pd
import numpy as np

df_customers = pd.read_csv(r"C:\Users\media\OneDrive\Desktop\My project-Ecommerce-kshastra\customers.csv")
df_inventory = pd.read_csv(r"C:\Users\media\OneDrive\Desktop\My project-Ecommerce-kshastra\inventory_snapshots.csv")
df_meta = pd.read_csv(r"C:\Users\media\OneDrive\Desktop\My project-Ecommerce-kshastra\meta_ads_campaigns.csv")
df_line_items = pd.read_csv(r"C:\Users\media\OneDrive\Desktop\My project-Ecommerce-kshastra\order_line_items.csv")
df_orders = pd.read_csv(r"C:\Users\media\OneDrive\Desktop\My project-Ecommerce-kshastra\orders.csv")
df_purchase = pd.read_csv(r"C:\Users\media\OneDrive\Desktop\My project-Ecommerce-kshastra\purchase_orders.csv")
df_catalog = pd.read_csv(r"C:\Users\media\OneDrive\Desktop\My project-Ecommerce-kshastra\sku_catalog.csv")
df_sessions = pd.read_csv(r"C:\Users\media\OneDrive\Desktop\My project-Ecommerce-kshastra\website_daily.csv")
df_daily = pd.read_csv(r"C:\Users\media\OneDrive\Desktop\My project-Ecommerce-kshastra\website_sessions.csv")

2.Customers

In [184]:
df_customers.head()

,Customer ID,Name,First order date,Total orders,Total revenue,Average order value,Time to 2nd purchase,Last purchase date,City / tier,Acquisition channel (first touch),RePurchased
0,CUST-0000001,Tejas Sarkar,2025-09-08,1,3356.0,3356.0,NaN,2025-09-08,Mumbai / Tier 1,Meta,N
1,CUST-0000003,Lila Bedi,2023-09-11,1,2360.0,2360.0,NaN,2023-09-11,Bengaluru / Tier 1,Google,N
2,CUST-0000004,Hamsini Parekh,2022-03-18,2,4029.0,2014.5,449.0,2023-06-11,Hyderabad / Tier 1,Google,Y
3,CUST-0000005,Yug Gill,2025-10-16,1,2581.0,2581.0,NaN,2025-10-16,Delhi / Tier 1,Offline (Friends/Family),N
4,CUST-0000007,Daksha Ramaswamy,2024-03-11,2,5365.0,2682.5,303.0,2025-01-08,Bengaluru / Tier 1,Meta,Y


In [185]:
df_customers[['city', 'tier']] = df_customers['City / tier'].str.split('/', expand=True)

In [186]:
df_customers['city'] = df_customers['city'].str.strip().str.title()
df_customers['tier'] = df_customers['tier'].str.strip()

In [187]:
df_customers.drop(columns=['City / tier'], inplace=True)

In [188]:
df_customers.rename(columns={
    "Customer ID": "customer_id", "Name": "customer_name", 
    "First order date": "first_order_date", "Total orders": "total_orders",
    "Total revenue": "total_revenue", "Average order value": "aov",
    "Time to 2nd purchase": "days_to_second_purchase", "Last purchase date": "last_purchase_date",
    "Acquisition channel (first touch)": "acquisition_channel", "RePurchased": "is_repeat_customer"
}, inplace=True)

3.Inventory Snapshots Cleaning

In [189]:
df_inventory.head()

,SKU,Category,Size,Units in stock,Units sold (last 7/30/60 days),Days of inventory left,Dead stock flag,date
0,SKU-SHI-0001,shirts,S,36,12/12/12,90.0,N,2022-01-02
1,SKU-PAN-0002,pants,XL,37,9/9/9,123.3,N,2022-01-02
2,SKU-SHI-0003,shirts,XXL,61,7/7/7,261.4,N,2022-01-02
3,SKU-TSH-0004,tshirts,XXL,118,2/2/2,999.0,N,2022-01-02
4,SKU-CRO-0005,croptops,S,53,3/3/3,530.0,N,2022-01-02


In [190]:
df_inventory.columns = df_inventory.columns.str.strip()
df_inventory['Category'] = df_inventory['Category'].str.strip().str.title()

In [191]:
df_inventory[['sold_7_days', 'sold_30_days', 'sold_60_days']] = df_inventory['Units sold (last 7/30/60 days)'].str.split('/', expand=True).astype(int)

In [192]:
df_inventory.rename(columns={
    "SKU": "sku_id", "Category": "category", "Size": "size", 
    "Units in stock": "units_in_stock", "Days of inventory left": "days_left_stock",
    "Dead stock flag": "is_dead_stock", "date": "snapshot_date"
}, inplace=True)

In [193]:
df_inventory.drop_duplicates(subset=['sku_id', 'snapshot_date'], inplace=True)

4.Order Line Items Cleaning

In [194]:
df_line_items.head()

,Order ID,SKU ID,"Category (top, bottom, outerwear, etc.)",Size,Color,MRP,Selling price,Discount %,Returned? (Y/N),Return reason (if any)
0,ORD-20220101-000001,SKU-PAN-0002,pants,XL,Maroon,2699,1814,32.8,N,NaN
1,ORD-20220101-000002,SKU-SHI-0001,shirts,XXL,Maroon,1999,1812,9.4,N,NaN
2,ORD-20220101-000003,SKU-CRO-0019,croptops,L,Olive,1799,2210,0.0,N,NaN
3,ORD-20220101-000004,SKU-SHI-0003,shirts,L,Black,2799,1395,50.0,Y,RTO
4,ORD-20220101-000005,SKU-SHI-0045,shirts,XS,Maroon,2299,999,50.0,N,NaN


In [195]:
df_line_items.columns = df_line_items.columns.str.strip()

In [196]:
df_line_items['Return reason (if any)'] = df_line_items['Return reason (if any)'].fillna('No Return')

In [197]:
df_line_items.rename(columns={
    "Order ID": "order_id", 
    "SKU ID": "sku_id", 
    "Category (top, bottom, outerwear, etc.)": "category",
    "MRP": "mrp", 
    "Selling price": "selling_price", 
    "Discount %": "item_discount_percent",
    "Returned? (Y/N)": "is_returned", 
    "Return reason (if any)": "return_reason"
}, inplace=True)

In [198]:
df_line_items['category'] = df_line_items['category'].str.title()

In [199]:
df_line_items['shipping_fees'] = np.where(df_line_items['selling_price'] > df_line_items['mrp'], 
df_line_items['selling_price'] - df_line_items['mrp'], 0)

df_line_items['clean_selling_price'] = df_line_items[['selling_price', 'mrp']].min(axis=1)

5.Meta Ads Campaigns Cleaning

In [200]:
df_meta.head()

,date,campaign_name,adset_name,Results,Amount spent (INR),spend,Reach,impressions,frequency,link_clicks,ctr_link,add_to_cart,initiate_checkout,purchases,purchase_conversion_value,cac,roas,creative_type,launch_date,Hook Rate
0,2023-01-01,KS_Jan23_Camp_01,Adset_Broad_18-24,8440,158966,158966,455462,970135,2.13,8440,0.0087,699,336,193,411436.11,823.66,2.59,UGC-Story,2023-01-08,0.217
1,2023-01-01,KS_Jan23_Camp_02,Adset_Interest_Streetwear,9364,119731,119731,493739,859107,1.74,9364,0.0109,638,295,152,400638.68,787.70,3.35,Static,2023-01-14,0.255
2,2023-03-01,KS_Mar23_Camp_03,Adset_Broad_18-24,2269,17562,17562,290910,378183,1.30,2269,0.0060,117,49,80,227457.17,219.52,12.95,UGC-Reel,2023-03-08,0.208
3,2023-03-01,KS_Mar23_Camp_04,Adset_Interest_Streetwear,4194,38895,38895,290095,699131,2.41,4194,0.0060,237,119,186,422317.36,209.11,10.86,UGC-Story,2023-03-01,0.168
4,2023-03-01,KS_Mar23_Camp_05,Adset_Retarget_7D,6369,36004,36004,456586,684880,1.50,6369,0.0093,418,188,166,369548.78,216.89,10.26,Studio-Reel,2023-03-05,0.181


In [201]:
df_meta = df_meta[::-1].reset_index(drop=True)
df_meta.drop(columns=['Amount spent (INR)'], inplace=True, errors='ignore')

In [202]:
df_meta.rename(columns={
    "Results": "results", "Reach": "reach", "link_clicks": "clicks",
    "ctr_link": "ctr", "initiate_checkout": "checkout", 
    "purchase_conversion_value": "revenue", "Hook Rate": "hook_rate"
}, inplace=True)

In [203]:
check_mask = (df_meta['impressions'].astype(float) >= df_meta['clicks'].astype(float)) & \
(df_meta['clicks'].astype(float) >= df_meta['purchases'].astype(float))
df_meta = df_meta[check_mask].copy()


df_meta = df_meta[check_mask].copy()


6.Orders Dataset Cleaning

In [204]:
df_orders.head()

,Order ID,Customer ID,Order date & time,Product,"Order value (gross, net)",Order value (gross),Order value (net),Discount applied (₹ + %),Discount applied (₹),Discount applied (%),Payment mode,Shipping city,Pincode,First order vs repeat,Channel source (last touch),Delivered / Returned / RTO
0,ORD-20220101-000001,CUST-0001627,2022-01-01 20:18:19,2-item cart,"2339,1814",2339,1814,"525,22.5%",525,22.5%,Card,Lucknow,624797,First,Meta,Delivered
1,ORD-20220101-000002,CUST-0006543,2022-01-01 23:42:40,2-item cart,"2588,1812",2588,1812,"777,30.0%",777,30.0%,UPI,Delhi,779192,First,Meta,Delivered
2,ORD-20220101-000003,CUST-0021661,2022-01-01 22:25:30,2-item cart,"2546,2210",2546,2210,"336,13.2%",336,13.2%,UPI,Chandigarh,516352,First,Organic Instagram,Delivered
3,ORD-20220101-000004,CUST-0007414,2022-01-01 10:16:28,2-item cart,"1800,1395",1800,1395,"405,22.5%",405,22.5%,COD,Indore,268472,First,Influencer,RTO
4,ORD-20220101-000005,CUST-0019325,2022-01-01 22:34:26,2-item cart,"2155,1853",2155,1853,"302,14.0%",302,14.0%,UPI,Mumbai,515911,First,Meta,Delivered


In [205]:
df_orders.drop(columns=['Product', 'Order value (gross, net)', 'Discount applied (₹ + %)'], inplace=True, errors='ignore')

In [206]:
df_orders.rename(columns={
    "Order ID": "order_id", "Customer ID": "customer_id", "Order date & time": "order_datetime",
    "Order value (gross)": "gross_value", "Order value (net)": "net_value", 
    "Discount applied (₹)": "discount_value", "Discount applied (%)": "discount_pct",
    "Payment mode": "payment_mode", "Shipping city": "city", "Pincode": "pincode",
    "First order vs repeat": "is_first_order", "Channel source (last touch)": "channel", "Delivered / Returned / RTO": "order_status"
}, inplace=True)

In [207]:
print(df_orders.columns.tolist())

['order_id', 'customer_id', 'order_datetime', 'gross_value', 'net_value', 'discount_value', 'discount_pct', 'payment_mode', 'city', 'pincode', 'is_first_order', 'channel', 'order_status']


In [208]:
df_orders['order_datetime'] = pd.to_datetime(df_orders['order_datetime'])

In [209]:
df_orders['order_date'] = df_orders['order_datetime'].dt.date
df_orders['order_time'] = df_orders['order_datetime'].dt.time
df_orders['order_month'] = df_orders['order_datetime'].dt.strftime('%Y-%m')

In [210]:
df_orders['discount_pct'] = df_orders['discount_pct'].replace('%', '', regex=True).astype(float)

In [211]:
df_orders['calculated_net_value'] = df_orders['gross_value'] - df_orders['discount_value']

In [212]:
df_orders['difference'] = df_orders['net_value'] - df_orders['calculated_net_value']

In [213]:
df_orders['city'] = df_orders['city'].str.strip().str.title()

7.Purchase Orders Cleaning

In [222]:
df_purchase.head()

,SKU,Vendor,Order quantity,Cost per unit,Order date,Expected delivery,Actual delivery,Lead time
0,SKU-SHI-0001,Urbanweave Co.,93,705,2022-01-09,2022-03-02,2022-03-07,57
1,SKU-PAN-0002,Apex Fashions,87,982,2022-01-16,2022-03-10,2022-03-10,53
2,SKU-SHI-0003,Knitcraft Studio,93,1004,2022-01-23,2022-03-01,2022-03-03,39
3,SKU-SHI-0001,Urbanweave Co.,351,705,2022-02-06,2022-04-01,2022-04-06,59
4,SKU-PAN-0002,Apex Fashions,136,982,2022-02-13,2022-03-19,2022-03-22,37


In [223]:
df_purchase.rename({'SKU':'sku_id','Vendor':'vendor','Order quantity':'quantity','Cost per unit':'cost_per_unit','Order date':'order_date','Expected delivery':'expected_delivery','Actual delivery':'actual_delivery','Lead time':'lead_time'}, inplace=True)

In [229]:
print(df_purchase.columns.tolist())

['SKU', 'Vendor', 'Order quantity', 'Cost per unit', 'Order date', 'Expected delivery', 'Actual delivery', 'Lead time']


In [231]:
df_purchase['Vendor'] = df_purchase['Vendor'].str.title()

In [233]:
df_purchase['total_cost_calc'] = df_purchase['Order quantity'] * df_purchase['Cost per unit']

8.SKU Catalog Cleaning

In [234]:
df_catalog.head()

,SKU,Category,Vendor,MRP,Cost_per_unit
0,SKU-SHI-0001,shirts,UrbanWeave Co.,1999,705
1,SKU-PAN-0002,pants,Apex Fashions,2699,982
2,SKU-SHI-0003,shirts,KnitCraft Studio,2799,1004
3,SKU-TSH-0004,tshirts,StitchWorks India,1499,539
4,SKU-CRO-0005,croptops,StitchWorks India,2399,846


In [235]:
df_catalog.rename(columns={"SKU": "sku_id", "Category": "category", "Vendor": "vendor", "MRP": "mrp", "Cost_per_unit": "cost_per_unit"}, inplace=True)

In [236]:
df_catalog['category'] = df_catalog['category'].str.title()

9.Website Traffic Analysis Sessions Cleaning

In [237]:
df_sessions.head()

,date,traffic_source,campaign_name,device_category,sessions,product_views,add_to_cart,begin_checkout,purchases,revenue,conversion_rate,aov,country,city,purchased
0,2022-01-01,Meta,KS_AlwaysOn,mobile,83,394,6,3,6,10002.82,0.0723,1667.14,India,Pune,1
1,2022-01-01,Meta,KS_AlwaysOn,desktop,45,124,2,1,6,10159.17,0.1333,1693.20,India,Mumbai,1
2,2022-01-01,Meta,KS_AlwaysOn,tablet,4,16,0,0,0,0.00,0.0000,0.00,India,Bengaluru,0
3,2022-01-01,Google,NaN,mobile,40,110,2,1,1,1768.70,0.0250,1768.70,India,Ahmedabad,1
4,2022-01-01,Google,NaN,desktop,19,54,0,0,3,5194.30,0.1579,1731.43,India,Jaipur,1


In [238]:
df_sessions['city'] = df_sessions['city'].str.title()

In [239]:
df_sessions['campaign_name'] = df_sessions['campaign_name'].str.title()

In [240]:
df_daily.drop_duplicates(subset=['session_id'], inplace=True)

10.Website Traffic Analysis Daily Cleaning

In [241]:
df_daily.head()

,session_id,date,traffic_source,campaign_name,device_category,city,sessions,product_views,add_to_cart,begin_checkout,purchased,order_id,customer_id,revenue
0,SESS-20220101-ME-MO-713914,1/1/2022,Meta,KS_AlwaysOn,mobile,Pune,1,5,0,0,0,NaN,NaN,0
1,SESS-20220101-ME-MO-629948,1/1/2022,Meta,KS_AlwaysOn,mobile,Pune,1,3,1,0,0,NaN,NaN,0
2,SESS-20220101-ME-MO-108341,1/1/2022,Meta,KS_AlwaysOn,mobile,Pune,1,2,0,0,0,NaN,NaN,0
3,SESS-20220101-ME-MO-358252,1/1/2022,Meta,KS_AlwaysOn,mobile,Pune,1,5,0,0,0,NaN,NaN,0
4,SESS-20220101-ME-MO-715627,1/1/2022,Meta,KS_AlwaysOn,mobile,Pune,1,3,0,0,0,NaN,NaN,0


In [242]:
print(df_daily.columns.tolist())

['session_id', 'date', 'traffic_source', 'campaign_name', 'device_category', 'city', 'sessions', 'product_views', 'add_to_cart', 'begin_checkout', 'purchased', 'order_id', 'customer_id', 'revenue']


In [243]:
df_daily['conversion_rate_check'] = np.where(df_daily['sessions'] == 0,0,df_daily['purchased'] / df_daily['sessions'])

In [244]:
df_daily['aov_check'] = np.where(df_daily['purchased'] == 0, 0,df_daily['revenue'] / df_daily['purchased'])

In [245]:
df_daily['campaign_name'] = df_daily['campaign_name'].fillna('no_campaign').str.title()

11.Final Export

In [246]:
import os
if not os.path.exists('cleaned_data'):
    os.makedirs('cleaned_data')

df_customers.to_csv('cleaned_data/customers_final.csv', index=False)
df_inventory.to_csv('cleaned_data/inventory_final.csv', index=False)
df_meta.to_csv('cleaned_data/meta_ads_final.csv', index=False)
df_line_items.to_csv('cleaned_data/line_items_final.csv', index=False)
df_orders.to_csv('cleaned_data/orders_final.csv', index=False)
df_purchase.to_csv('cleaned_data/purchase_final.csv', index=False)
df_catalog.to_csv('cleaned_data/catalog_final.csv', index=False)
df_sessions.to_csv('cleaned_data/sessions_final.csv', index=False)
df_daily.to_csv('cleaned_data/daily_final.csv', index=False)
